In [32]:
import numpy as np 
import pandas as pd

In [33]:
df = pd.read_csv('/kaggle/input/news-headlines-dataset/labelled_train_set.csv')
df = df[['News/Comment','Type']]
df.head(10)

,News/Comment,Type
0,കേള്‍വി തകരാറുള്ള കുട്ടികള്‍ക്ക് നടത്തുന്ന സൗജ...,FALSE
1,ചന്ദ്രയാന് കേരള മുഖ്യമന്ത്രി പിണറായി വിജയൻ മാത...,FALSE
2,പിണറായി വിജയന്‍ സര്‍ക്കാര്‍ നിര്‍മിച്ച കേരളത്ത...,FALSE
3,മുഖ്യമന്ത്രിയുടെ ബിനാമി എന്ന് സ്വർണക്കടത്തു കേ...,FALSE
4,പിണറായി വിജയന്‍ ഇടപെട്ട് കേരളത്തില്‍ നിന്നുള്ള...,FALSE
5,മുഖ്യമന്ത്രി പിണറായി വിജയനെ ഡല്‍ഹിയിലേക്ക് വിള...,FALSE
6,കോടതി പറഞ്ഞാല്‍ പൗരത്വ നിയമം നടപ്പിലാക്കുമെന്ന...,FALSE
7,"""അയ്യപ്പ വിശ്വാസികളുടെ വോട്ട് സി.പി.എമ്മിന് വേ...",FALSE
8,ഏതൊരു മലയാളിയേയും പോലെ പിണറായി സർക്കാരിന്റെ തു...,FALSE
9,മുഖ്യമന്ത്രി പിണറായി വിജയന്‍ ധര്‍മടം മണ്ഡലത്തി...,FALSE


In [34]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [4]:
model_name = 'ai4bharat/indic-bert'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=5)

config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

2025-04-25 19:26:53.082942: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745609213.274822      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745609213.324247      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

In [35]:
label2id = {
    "FALSE": 0,
    "HALF TRUE": 1,
    "MOSTLY FALSE": 2,
    "PARTLY FALSE": 3,
    "MOSTLY TRUE": 4
}

In [36]:
from torch.nn import CrossEntropyLoss 
import torch

In [37]:
new_row = {
    "News/Comment" : "ഈ വാർത്ത കൂടുതലായും ശരിയാകാം",
    'Type' : "MOSTLY TRUE"
}
df = pd.concat([df,pd.DataFrame([new_row])],ignore_index=True)

In [38]:
label_order = ["FALSE", "HALF TRUE", "MOSTLY FALSE", "PARTLY FALSE", "MOSTLY TRUE"]
value_counts = df['Type'].value_counts()
class_counts = [value_counts.get(label,0) for label in label_order]
print(class_counts)

[1130, 138, 230, 38, 2]


In [39]:
total = sum(class_counts)
weights = [total/c for c in class_counts]
weights = torch.tensor(weights,dtype=torch.float)
loss_fn = CrossEntropyLoss(weight=weights)

In [40]:
import pandas as pd 
from transformers import TrainingArguments
from sklearn.model_selection import train_test_split 
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
import numpy as np 
from sklearn.metrics import classification_report,accuracy_score,f1_score

In [41]:

id2label = {v:k for k,v in label2id.items()}
df['label'] = df['Type'].map(label2id)

In [63]:
train_df,val_df = train_test_split(df,test_size=0.1,stratify=df['label'],random_state=42)
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

In [31]:
lens = []
for i in df['News/Comment']:
    lens.append(len(str(i)))
print(max(lens))

282


In [52]:
def tokenize(batch):
    texts = [str(x) for x in batch['News/Comment']]
    return tokenizer(texts,
                     padding='max_length',
                     truncation=True,
                     max_length=512)

In [64]:
train_dataset = train_dataset.map(tokenize,batched=True)
val_dataset = val_dataset.map(tokenize,batched=True)

Map:   0%|          | 0/1384 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

In [66]:
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [55]:
training_args = TrainingArguments(
    output_dir = '/results',
    eval_strategy="epoch",
    save_strategy ='epoch',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs = 10,
    weight_decay = 0.01,
    learning_rate = 2e-5,
    logging_dir = '/logs',
    logging_steps = 1,
    load_best_model_at_end = True,
    metric_for_best_model='f1',
    save_total_limit = 20,
    report_to="none"
)

In [56]:
def compute_metrics(eval_pred):
    logits,labels = eval_pred
    preds = torch.argmax(torch.tensor(logits),dim=1)
    acc = accuracy_score(labels,preds)
    f1 = f1_score(labels,preds,average='weighted')
    return {'accuracy':acc,'f1':f1}
    

In [57]:
from transformers import Trainer

In [58]:
class CustomTrainer(Trainer):
    def compute_loss(self,model,inputs,return_outputs=False):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = loss_fn(logits,labels)
        return (loss,outputs) if return_outputs else loss

In [59]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics,
)

/tmp/ipykernel_31/308854565.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [60]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.607600,0.877136,0.720779,0.684887
2,0.489800,0.895415,0.740260,0.669970
3,0.296700,1.035990,0.681818,0.652429
4,0.246100,1.192608,0.688312,0.664346
5,0.708800,1.255075,0.668831,0.649530
6,0.035300,1.305791,0.714286,0.691067
7,0.008200,1.400517,0.707792,0.685111
8,0.040500,1.443917,0.688312,0.670592
9,0.015300,1.514774,0.707792,0.683413
10,0.200500,1.539051,0.707792,0.683413


TrainOutput(global_step=870, training_loss=0.2987993291268746, metrics={'train_runtime': 762.9295, 'train_samples_per_second': 18.141, 'train_steps_per_second': 1.14, 'total_flos': 330847813877760.0, 'train_loss': 0.2987993291268746, 'epoch': 10.0})

In [68]:
test_df = pd.read_csv('/kaggle/input/news-headlines-dataset/unlabelled_test1.csv')
texts = test_df["News/Comment"].tolist() 

In [71]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

AlbertForSequenceClassification(
  (albert): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(200000, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertSdpaAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features

In [72]:

tokenized_inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
tokenized_inputs = {k: v.to(device) for k, v in tokenized_inputs.items()}
model.eval()
with torch.no_grad():
    outputs = model(**tokenized_inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
predicted_labels = [id2label[pred] for pred in predictions]
test_df["predicted_label"] = predicted_labels

In [74]:
test_df

,ID,News/Comment,predicted_label
0,TEST_1,വിഴിഞ്ഞത്ത് തീരദേശവാസികള്‍ ആക്രമിച്ചപ്പോള്‍ മു...,FALSE
1,TEST_2,കുരിശിന് മുന്നില്‍ കൈകൂപ്പി നില്‍ക്കുന്ന പിണറാ...,MOSTLY FALSE
2,TEST_3,1971ലെ തലശ്ശേരി കലാപത്തില്‍ പങ്കുണ്ടായിരുന്ന പ...,FALSE
3,TEST_4,ഗവര്‍ണര്‍ ആരിഫ് മുഹമ്മദ് ഖാനുമായുള്ളതര്‍ക്കം പ...,HALF TRUE
4,TEST_5,എല്ലാ മേഖലകളിലും കേരളം തകര്‍ന്നടിഞ്ഞുവെന്ന് സമ...,FALSE
...,...,...,...
127,TEST_128,ന്യൂയോർക്ക് ടൈംസിന്റെ എഡിറ്റർ ജോസഫ് ഹോപ്പ് ഒരു...,FALSE
128,TEST_129,ഷിർദിയിലെ സായി ക്ഷേത്രത്തിൽ ഹിന്ദുക്കൾ നൽകുന്ന...,FALSE
129,TEST_130,ഉത്തര്പ്രദേശിലെ മദ്രസയില് യുവാവ് കുട്ടിയെ ക്രൂ...,FALSE
130,TEST_131,ഐഎസ്ആർഒ ചന്ദ്രയാൻ -3 മിഷൻ ശനിയിൽ ഇറങ്ങുന്നതിന്...,FALSE
